# Extract and Rank Misrecognized Words from ASR Evaluations

In [ ]:
!pip install -U -q jiwer pandas tqdm

## Load the Evaluation Results (TSV Files from the ASR Exercise)

In [ ]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path("../mmai_07")

tsv_paths = sorted(RESULTS_DIR.glob("*without_context.tsv"))
print(tsv_paths)

results = []
for tsv_path in tsv_paths:
    df = pd.read_csv(tsv_path, sep="\t")
    df["source"] = tsv_path.stem
    results.append(df)

results = pd.concat(results, ignore_index=True)
results.head()

## Text Normalization and Word-Level Alignment

In [ ]:
import jiwer
import re
import string

def normalize(text):
    text = str(text).translate(str.maketrans("", "", string.punctuation)).lower().strip()
    text = re.sub(r" +", " ", text).strip()
    return text

def word_errors(reference, hypothesis):
    """(error_type, ref_words, hyp_words) tuples from the jiwer alignment."""
    ref, hyp = normalize(reference), normalize(hypothesis)
    out = []
    for c in jiwer.process_words(ref, hyp).alignments[0].chunks:
        r = " ".join(ref.split()[c.ref_start_idx:c.ref_end_idx])
        h = " ".join(hyp.split()[c.hyp_start_idx:c.hyp_end_idx])
        if c.type != "equal":
            out.append((c.type, r, h))
    return out

word_errors(results.iloc[0]["label"], results.iloc[0]["prediction"])

## Collect and Rank the Problem Words

In [ ]:
from collections import Counter

word_counter = Counter()
examples = {}

for _, row in results.iterrows():
    for etype, r, h in word_errors(row["label"], row["prediction"]):
        for w in r.split():
            if len(w) < 4:   # skip very short words
                continue
            word_counter[w] += 1
            examples.setdefault(w, {"label": row["label"], "prediction": row["prediction"],
                                    "source": row["source"]})

problem_words = pd.DataFrame(
    [(w, c, examples[w]["label"], examples[w]["prediction"], examples[w]["source"])
     for w, c in word_counter.most_common()],
    columns=["word", "error_count", "example_label", "example_prediction", "example_source"],
)
problem_words.head(20)

## Save the Ranked Word List

In [ ]:
problem_words.to_csv("tts_exercise/problem_words.csv", sep="\t", index=False)
print(f"Saved {len(problem_words)} problem words to tts_exercise/problem_words.csv")